<a href="https://colab.research.google.com/github/renishpatel-12/code-alpha-internship-ml-tasks/blob/main/CodeAlpha_SpeechEmotionRecognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install librosa
!pip install soundfile
!pip install tensorflow

In [2]:
import os
import librosa
import numpy as np
import pandas as pd

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dropout

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Path to dataset files: /kaggle/input/ravdess-emotional-speech-audio


In [5]:
from google.colab import files

uploaded = files.upload()

Saving 03-01-01-01-01-01-01.wav to 03-01-01-01-01-01-01.wav


In [6]:
def extract_features(file_name):

    audio, sample_rate = librosa.load(
        file_name,
        duration=3,
        offset=0.5
    )

    mfccs = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=40
    )

    mfccs_scaled = np.mean(
        mfccs.T,
        axis=0
    )

    return mfccs_scaled

In [13]:
features = []
labels = []

for root, dirs, files in os.walk("/kaggle/input/ravdess-emotional-speech-audio"):

    for file in files:

        if file.endswith(".wav"):

            file_path = os.path.join(root,file)

            emotion = file.split("-")[2]

            data = extract_features(file_path)

            features.append(data)

            labels.append(emotion)

In [14]:
X = np.array(features)
y = np.array(labels)

In [15]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

In [18]:
model = Sequential()

model.add(
    LSTM(
        128,
        return_sequences=False,
        input_shape=(40,1)
    )
)

model.add(Dropout(0.3))

model.add(Dense(64,activation='relu'))

model.add(Dense(
    len(np.unique(y)),
    activation='softmax'
))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [19]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [20]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test,y_test)
)

Epoch 1/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.2031 - loss: 2.0097 - val_accuracy: 0.2830 - val_loss: 1.8515
Epoch 2/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2839 - loss: 1.8672 - val_accuracy: 0.2899 - val_loss: 1.8082
Epoch 3/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3090 - loss: 1.8050 - val_accuracy: 0.3108 - val_loss: 1.7702
Epoch 4/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3181 - loss: 1.7888 - val_accuracy: 0.3247 - val_loss: 1.7343
Epoch 5/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3307 - loss: 1.7549 - val_accuracy: 0.3299 - val_loss: 1.7408
Epoch 6/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3333 - loss: 1.7322 - val_accuracy: 0.3333 - val_loss: 1.7235
Epoch 7/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3411 - loss: 1.7139 - val_accuracy: 0.3420 - val_loss: 1.7585
Epoch 8/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3372 - loss: 1.6997 - val_accuracy: 0.3507 - val_loss

In [21]:
loss, acc = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:",acc)

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7708 - loss: 0.7811
Accuracy: 0.7708333134651184


In [22]:
model.save(
    "speech_emotion_model.h5"
)

In [23]:
from google.colab import files

files.download(
    "speech_emotion_model.h5"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>